# RealWaste Segmentation Preparation

Standalone Kaggle notebook for preparing RealWaste into the same normalized COCO-like format used by the merge pipeline.

The notebook discovers a class-folder RealWaste dataset under `/kaggle/input`, maps RealWaste classes into the AquaTrash 8-class label space, then synthesizes one polygon mask per image using either SAM or your trained YOLO segmentation model. Outputs are written to `/kaggle/working/data/normalized/realwaste`.


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import shutil
import subprocess
import sys

INSTALL_MISSING_PACKAGES = True
if INSTALL_MISSING_PACKAGES:
    required_packages = [
        ("ultralytics", "ultralytics>=8.0"),
        ("PIL", "pillow>=10.0"),
        ("matplotlib", "matplotlib>=3.7"),
    ]
    missing_packages = [package for module, package in required_packages if importlib.util.find_spec(module) is None]
    if missing_packages:
        print("Installing missing packages:", missing_packages)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])
    else:
        print("Dependencies are already installed.")

In [ ]:
from PIL import Image

IS_KAGGLE = Path("/kaggle").exists()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path("data/raw")
WORKING_DIR = Path("/kaggle/working") if IS_KAGGLE else Path(".")

# Leave as None to auto-discover a folder containing RealWaste class directories.
REALWASTE_ROOT_OVERRIDE = None

# Toggle mask source: "sam" or "yolo".
SEGMENTATION_MODEL = os.getenv("SEGMENTATION_MODEL", "sam").lower()

# SAM settings. High-accuracy checkpoint supported by Ultralytics assets.
# If Kaggle GPU memory is tight, switch to "sam2.1_b.pt" or "sam_l.pt"; for speed, use "mobile_sam.pt".
SAM_MODEL = "sam2.1_l.pt"
SAM_WEIGHTS_DIR = WORKING_DIR / "models" / "sam"
SAM_IMGSZ = 1024
SAM_POINTS = "center_plus_quarters"

# YOLO settings. Set YOLO_MODEL_PATH to your trained segmentation weights, for example:
# Path("/kaggle/input/my-yolo-run/best.pt"). If None, the notebook tries to discover a best*.pt under /kaggle/input.
YOLO_MODEL_PATH = None
YOLO_CONF = 0.25
YOLO_IMGSZ = 640
YOLO_FALLBACK_TO_SAM = False

FORCE_REGENERATE_MASKS = False
SAVE_CACHE_EVERY = 25
MAX_IMAGES = None

OUTPUT_DIR = WORKING_DIR / "data" / "normalized" / "realwaste"
CACHE_PATH = OUTPUT_DIR / "segmentation_cache.json"
ANNOTATIONS_PATH = OUTPUT_DIR / "annotations.json"
SUMMARY_PATH = OUTPUT_DIR / "realwaste_prepare_summary.json"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXPECTED_CLASSES = {
    "Cardboard",
    "Food Organics",
    "Glass",
    "Metal",
    "Miscellaneous Trash",
    "Paper",
    "Plastic",
    "Textile Trash",
    "Vegetation",
}

AQUATRASH_CATEGORIES = [
    {"id": 1, "name": "glass"},
    {"id": 2, "name": "metal_can"},
    {"id": 3, "name": "paper_cardboard"},
    {"id": 4, "name": "plastic_battle"},
    {"id": 5, "name": "plastic_bag"},
    {"id": 6, "name": "rigid_plastic"},
    {"id": 7, "name": "organic_waste"},
    {"id": 8, "name": "mixed_waste"},
]

REALWASTE_TO_AQUATRASH_LABEL = {
    "Cardboard": "paper_cardboard",
    "Food Organics": "organic_waste",
    "Glass": "glass",
    "Metal": "metal_can",
    "Miscellaneous Trash": "mixed_waste",
    "Paper": "paper_cardboard",
    "Plastic": "rigid_plastic",
    "Textile Trash": "mixed_waste",
    "Vegetation": "organic_waste",
}

print("IS_KAGGLE:", IS_KAGGLE)
print("INPUT_ROOT:", INPUT_ROOT)
print("WORKING_DIR:", WORKING_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("SEGMENTATION_MODEL:", SEGMENTATION_MODEL)
print("SAM_MODEL:", SAM_MODEL)
print("SAM_WEIGHTS_DIR:", SAM_WEIGHTS_DIR)
print("SAM_IMGSZ:", SAM_IMGSZ)
print("YOLO_MODEL_PATH:", YOLO_MODEL_PATH)
print("YOLO_CONF:", YOLO_CONF)
print("YOLO_IMGSZ:", YOLO_IMGSZ)
print("YOLO_FALLBACK_TO_SAM:", YOLO_FALLBACK_TO_SAM)

if INPUT_ROOT.exists():
    print("\nInput roots:")
    for path in sorted(INPUT_ROOT.iterdir()):
        print(" -", path)


In [ ]:
# Load the selected segmentation model before the image-processing loop starts.
# For SAM, this downloads the checkpoint explicitly. For YOLO, point YOLO_MODEL_PATH at your trained best.pt.
from ultralytics import SAM, YOLO
from ultralytics.utils.downloads import ASSETS_URL, GITHUB_ASSETS_NAMES, safe_download

SEGMENTATION_MODEL = SEGMENTATION_MODEL.lower().strip()
if SEGMENTATION_MODEL not in {"sam", "yolo"}:
    raise ValueError("SEGMENTATION_MODEL must be either 'sam' or 'yolo'.")

SAM_MODEL_INSTANCE = None
YOLO_MODEL_INSTANCE = None
SAM_MODEL_PATH = None
YOLO_MODEL_PATH_RESOLVED = None


def resolve_sam_model_path() -> Path:
    SAM_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
    sam_model_path = Path(SAM_MODEL)
    if sam_model_path.exists():
        return sam_model_path.resolve()
    if SAM_MODEL in GITHUB_ASSETS_NAMES:
        resolved = SAM_WEIGHTS_DIR / SAM_MODEL
        if not resolved.exists():
            print(f"Downloading SAM checkpoint {SAM_MODEL} to {resolved}...")
            safe_download(
                url=f"{ASSETS_URL}/{SAM_MODEL}",
                file=resolved,
                min_bytes=1_000_000,
                exist_ok=True,
                progress=True,
            )
        else:
            print(f"SAM checkpoint already exists: {resolved}")
        return resolved
    supported = sorted(name for name in GITHUB_ASSETS_NAMES if "sam" in name.lower())
    raise ValueError(
        f"Unsupported SAM_MODEL '{SAM_MODEL}'. Supported Ultralytics SAM assets include: {supported}. "
        "Set SAM_MODEL to one of these names or to a local .pt/.pth path."
    )


def discover_yolo_model_path() -> Path:
    if YOLO_MODEL_PATH:
        candidate = Path(YOLO_MODEL_PATH)
        if not candidate.exists():
            raise FileNotFoundError(f"YOLO_MODEL_PATH does not exist: {candidate}")
        return candidate.resolve()

    search_roots = [INPUT_ROOT, WORKING_DIR]
    patterns = ["**/best.pt", "**/*best*.pt", "**/*.pt"]
    candidates = []
    sam_like = ("sam", "fastsam", "mobile_sam")
    for root in search_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            for path in root.glob(pattern):
                lower_name = path.name.lower()
                if any(token in lower_name for token in sam_like):
                    continue
                candidates.append(path)
            if candidates:
                break
        if candidates:
            break

    if not candidates:
        raise FileNotFoundError(
            "Could not auto-discover YOLO segmentation weights. Set YOLO_MODEL_PATH to your trained best.pt, "
            "for example Path('/kaggle/input/my-yolo-run/best.pt')."
        )

    candidates = sorted(set(candidates), key=lambda p: ("best" not in p.name.lower(), len(str(p)), str(p)))
    print("Auto-discovered YOLO_MODEL_PATH:", candidates[0])
    return candidates[0].resolve()


if SEGMENTATION_MODEL == "sam":
    SAM_MODEL_PATH = resolve_sam_model_path()
    print(f"Loading SAM model checkpoint: {SAM_MODEL_PATH}")
    SAM_MODEL_INSTANCE = SAM(str(SAM_MODEL_PATH))
    try:
        SAM_MODEL_INSTANCE.info(verbose=True)
    except Exception as exc:
        print(f"SAM model loaded, but model info could not be printed: {type(exc).__name__}: {exc}")
    print("SAM model is ready.")
elif SEGMENTATION_MODEL == "yolo":
    YOLO_MODEL_PATH_RESOLVED = discover_yolo_model_path()
    print(f"Loading YOLO segmentation model: {YOLO_MODEL_PATH_RESOLVED}")
    YOLO_MODEL_INSTANCE = YOLO(str(YOLO_MODEL_PATH_RESOLVED))
    print("YOLO model is ready.")
    if YOLO_FALLBACK_TO_SAM:
        SAM_MODEL_PATH = resolve_sam_model_path()
        print(f"SAM fallback checkpoint is ready: {SAM_MODEL_PATH}")


In [ ]:
def discover_realwaste_root(input_root: Path = INPUT_ROOT, override: str | Path | None = REALWASTE_ROOT_OVERRIDE) -> Path:
    if override:
        root = Path(override)
        if not root.exists():
            raise FileNotFoundError(f"REALWASTE_ROOT_OVERRIDE does not exist: {root}")
        return root

    search_roots = [input_root]
    if not IS_KAGGLE:
        search_roots.extend([Path("data/raw"), Path("../data/raw")])

    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        paths = [root]
        paths.extend(path for path in root.glob("*") if path.is_dir())
        paths.extend(path for path in root.glob("*/*") if path.is_dir())
        for path in paths:
            try:
                child_dirs = {child.name for child in path.iterdir() if child.is_dir()}
            except OSError:
                continue
            score = len(child_dirs & EXPECTED_CLASSES)
            if score >= 5:
                candidates.append((score, -len(path.parts), path))

    if not candidates:
        raise FileNotFoundError("Could not find a RealWaste root containing the expected class folders.")

    candidates.sort(reverse=True)
    return candidates[0][2]


def safe_file_component(value: str) -> str:
    return "_".join(value.replace("-", "_").split())


def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def save_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)


def polygon_area(segmentation: list[float]) -> float:
    if len(segmentation) < 6:
        return 0.0
    points = list(zip(segmentation[0::2], segmentation[1::2]))
    area = 0.0
    for index, (x1, y1) in enumerate(points):
        x2, y2 = points[(index + 1) % len(points)]
        area += (x1 * y2) - (x2 * y1)
    return abs(area) / 2.0


def bbox_from_segmentation(segmentation: list[float]) -> list[float]:
    xs = segmentation[0::2]
    ys = segmentation[1::2]
    min_x = min(xs)
    min_y = min(ys)
    max_x = max(xs)
    max_y = max(ys)
    return [min_x, min_y, max_x - min_x, max_y - min_y]


def rectangle_segmentation(width: int, height: int) -> list[float]:
    max_x = float(max(width - 1, 1))
    max_y = float(max(height - 1, 1))
    return [0.0, 0.0, max_x, 0.0, max_x, max_y, 0.0, max_y]


def clamp_polygon(segmentation: list[float], width: int, height: int) -> list[float]:
    max_x = float(max(width - 1, 1))
    max_y = float(max(height - 1, 1))
    clamped = []
    for index in range(0, len(segmentation) - 1, 2):
        x = max(0.0, min(max_x, float(segmentation[index])))
        y = max(0.0, min(max_y, float(segmentation[index + 1])))
        clamped.extend([x, y])
    return clamped


def prompt_points(width: int, height: int) -> list[list[float]]:
    if SAM_POINTS == "center":
        return [[width / 2.0, height / 2.0]]
    return [
        [width / 2.0, height / 2.0],
        [width * 0.38, height / 2.0],
        [width * 0.62, height / 2.0],
        [width / 2.0, height * 0.38],
        [width / 2.0, height * 0.62],
    ]


def load_sam_model():
    global SAM_MODEL_INSTANCE, SAM_MODEL_PATH
    if "SAM_MODEL_INSTANCE" not in globals() or SAM_MODEL_INSTANCE is None:
        from ultralytics import SAM

        if "SAM_MODEL_PATH" not in globals() or SAM_MODEL_PATH is None:
            SAM_MODEL_PATH = resolve_sam_model_path()
        print(f"Loading SAM model checkpoint: {SAM_MODEL_PATH}")
        SAM_MODEL_INSTANCE = SAM(str(SAM_MODEL_PATH))
    return SAM_MODEL_INSTANCE


def load_yolo_model():
    global YOLO_MODEL_INSTANCE, YOLO_MODEL_PATH_RESOLVED
    if "YOLO_MODEL_INSTANCE" not in globals() or YOLO_MODEL_INSTANCE is None:
        from ultralytics import YOLO

        if "YOLO_MODEL_PATH_RESOLVED" not in globals() or YOLO_MODEL_PATH_RESOLVED is None:
            YOLO_MODEL_PATH_RESOLVED = discover_yolo_model_path()
        print(f"Loading YOLO segmentation model: {YOLO_MODEL_PATH_RESOLVED}")
        YOLO_MODEL_INSTANCE = YOLO(str(YOLO_MODEL_PATH_RESOLVED))
    return YOLO_MODEL_INSTANCE


def extract_best_polygon_from_masks(masks, width: int, height: int) -> tuple[list[float] | None, float]:
    best_segmentation = None
    best_area = 0.0
    if masks is None:
        return None, 0.0
    for polygon in masks.xy:
        segmentation = []
        for x, y in polygon:
            segmentation.extend([float(x), float(y)])
        segmentation = clamp_polygon(segmentation, width, height)
        area = polygon_area(segmentation)
        area_ratio = area / float(max(width * height, 1))
        if len(segmentation) >= 6 and 0.001 <= area_ratio <= 0.995 and area > best_area:
            best_segmentation = segmentation
            best_area = area
    return best_segmentation, best_area


def generate_sam_segmentation(sam_model, image_path: Path, width: int, height: int) -> tuple[list[float], str]:
    points = prompt_points(width, height)
    results = sam_model.predict(
        source=str(image_path),
        points=points,
        labels=[1] * len(points),
        imgsz=SAM_IMGSZ,
        retina_masks=True,
        verbose=False,
        save=False,
    )

    if results:
        segmentation, _ = extract_best_polygon_from_masks(getattr(results[0], "masks", None), width, height)
        if segmentation is not None:
            return segmentation, "sam"
    return rectangle_segmentation(width, height), "fallback"


def generate_yolo_segmentation(yolo_model, image_path: Path, width: int, height: int) -> tuple[list[float], str]:
    results = yolo_model.predict(
        source=str(image_path),
        conf=YOLO_CONF,
        imgsz=YOLO_IMGSZ,
        retina_masks=True,
        verbose=False,
        save=False,
    )
    if results:
        segmentation, _ = extract_best_polygon_from_masks(getattr(results[0], "masks", None), width, height)
        if segmentation is not None:
            return segmentation, "yolo"

    if YOLO_FALLBACK_TO_SAM:
        sam_model = load_sam_model()
        segmentation, source = generate_sam_segmentation(sam_model, image_path, width, height)
        return segmentation, "sam_fallback" if source == "sam" else source
    return rectangle_segmentation(width, height), "fallback"


def generate_segmentation(image_path: Path, width: int, height: int) -> tuple[list[float], str]:
    if SEGMENTATION_MODEL == "sam":
        return generate_sam_segmentation(load_sam_model(), image_path, width, height)
    if SEGMENTATION_MODEL == "yolo":
        return generate_yolo_segmentation(load_yolo_model(), image_path, width, height)
    raise ValueError("SEGMENTATION_MODEL must be either 'sam' or 'yolo'.")


def segmentation_cache_model_key() -> str:
    if SEGMENTATION_MODEL == "sam":
        return f"sam:{globals().get('SAM_MODEL_PATH', SAM_MODEL)}:{SAM_IMGSZ}:{SAM_POINTS}"
    return f"yolo:{globals().get('YOLO_MODEL_PATH_RESOLVED', YOLO_MODEL_PATH)}:{YOLO_CONF}:{YOLO_IMGSZ}:fallback_sam={YOLO_FALLBACK_TO_SAM}"


In [ ]:
def collect_realwaste_images(root_dir: Path) -> list[tuple[str, Path]]:
    image_paths = []
    for class_name in sorted(EXPECTED_CLASSES):
        class_dir = root_dir / class_name
        if not class_dir.exists():
            print(f"Warning: class folder not found: {class_dir}")
            continue
        for image_path in sorted(class_dir.iterdir()):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                image_paths.append((class_name, image_path))
    if MAX_IMAGES is not None:
        image_paths = image_paths[:MAX_IMAGES]
    return image_paths


def prepare_realwaste_with_segmentation_model(root_dir: Path) -> dict:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    label_to_id = {category["name"]: category["id"] for category in AQUATRASH_CATEGORIES}
    cache = load_json(CACHE_PATH)
    image_paths = collect_realwaste_images(root_dir)
    if not image_paths:
        raise FileNotFoundError(f"No RealWaste images found under {root_dir}")

    cache_model_key = segmentation_cache_model_key()
    print(f"Preparing {len(image_paths)} RealWaste images from {root_dir}")
    print(f"SEGMENTATION_MODEL: {SEGMENTATION_MODEL}")
    print(f"Segmentation cache: {CACHE_PATH}")
    print(f"Cache model key: {cache_model_key}")
    images = []
    annotations = []
    counts_by_class = {class_name: 0 for class_name in sorted(EXPECTED_CLASSES)}
    masks_by_source = {"sam": 0, "yolo": 0, "sam_fallback": 0, "fallback": 0}

    for image_index, (class_name, image_path) in enumerate(image_paths, start=1):
        with Image.open(image_path) as image:
            width, height = image.size

        cache_key = image_path.relative_to(root_dir).as_posix()
        cached_mask = cache.get(cache_key)
        if (
            cached_mask is not None
            and not FORCE_REGENERATE_MASKS
            and cached_mask.get("model_key") == cache_model_key
            and int(cached_mask.get("width", width)) == width
            and int(cached_mask.get("height", height)) == height
        ):
            segmentation = cached_mask["segmentation"]
            mask_source = cached_mask.get("mask_source", SEGMENTATION_MODEL)
        else:
            segmentation, mask_source = generate_segmentation(image_path, width, height)
            cache[cache_key] = {
                "segmentation": segmentation,
                "mask_source": mask_source,
                "model_key": cache_model_key,
                "width": width,
                "height": height,
            }

        masks_by_source[mask_source] = masks_by_source.get(mask_source, 0) + 1

        output_file_name = f"{safe_file_component(class_name)}_{image_path.name.replace(' ', '_')}"
        shutil.copy2(image_path, OUTPUT_DIR / output_file_name)

        image_id = len(images) + 1
        annotation_id = len(annotations) + 1
        aquatrash_label = REALWASTE_TO_AQUATRASH_LABEL[class_name]
        category_id = label_to_id[aquatrash_label]
        area = polygon_area(segmentation)
        bbox = bbox_from_segmentation(segmentation)

        images.append({
            "id": image_id,
            "width": width,
            "height": height,
            "file_name": output_file_name,
            "source_file_name": cache_key,
        })
        annotations.append({
            "id": annotation_id,
            "image_id": image_id,
            "category_id": category_id,
            "segmentation": [segmentation],
            "area": area,
            "bbox": bbox,
            "iscrowd": 0,
        })
        counts_by_class[class_name] += 1

        if image_index % SAVE_CACHE_EVERY == 0:
            save_json(cache, CACHE_PATH)
        if image_index % 100 == 0:
            print(f"Prepared {image_index}/{len(image_paths)} masks")

    save_json(cache, CACHE_PATH)
    payload = {
        "images": images,
        "annotations": annotations,
        "categories": AQUATRASH_CATEGORIES,
        "info": {
            "source": "RealWaste",
            "segmentation_model": SEGMENTATION_MODEL,
            "sam_model": SAM_MODEL,
            "sam_model_path": str(globals().get("SAM_MODEL_PATH")),
            "sam_imgsz": SAM_IMGSZ,
            "sam_points": SAM_POINTS,
            "yolo_model_path": str(globals().get("YOLO_MODEL_PATH_RESOLVED")),
            "yolo_conf": YOLO_CONF,
            "yolo_imgsz": YOLO_IMGSZ,
            "yolo_fallback_to_sam": YOLO_FALLBACK_TO_SAM,
            "segmentation_cache": str(CACHE_PATH),
            "masks_by_source": masks_by_source,
        },
    }
    save_json(payload, ANNOTATIONS_PATH)

    summary = {
        "realwaste_root": str(root_dir),
        "output_dir": str(OUTPUT_DIR),
        "annotations_path": str(ANNOTATIONS_PATH),
        "segmentation_model": SEGMENTATION_MODEL,
        "sam_model": SAM_MODEL,
        "sam_model_path": str(globals().get("SAM_MODEL_PATH")),
        "yolo_model_path": str(globals().get("YOLO_MODEL_PATH_RESOLVED")),
        "images": len(images),
        "annotations": len(annotations),
        "masks_by_source": masks_by_source,
        "counts_by_class": counts_by_class,
        "target_categories": AQUATRASH_CATEGORIES,
    }
    save_json(summary, SUMMARY_PATH)
    return summary


In [ ]:
REALWASTE_ROOT = discover_realwaste_root()
print("REALWASTE_ROOT:", REALWASTE_ROOT)

summary = prepare_realwaste_with_segmentation_model(REALWASTE_ROOT)
print(json.dumps(summary, indent=2)[:4000])


In [ ]:
def visualize_prepared_samples(samples: int = 6) -> None:
    import matplotlib.pyplot as plt
    from matplotlib.patches import Polygon

    payload = load_json(ANNOTATIONS_PATH)
    annotations_by_image = {ann["image_id"]: ann for ann in payload.get("annotations", [])}
    categories = {cat["id"]: cat["name"] for cat in payload.get("categories", [])}
    images = payload.get("images", [])[:samples]
    if not images:
        print("No prepared images to visualize.")
        return

    cols = min(3, len(images))
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")

    for ax, image_info in zip(axes.flat, images):
        image_path = OUTPUT_DIR / image_info["file_name"]
        ann = annotations_by_image.get(image_info["id"])
        image = Image.open(image_path)
        ax.imshow(image)
        ax.set_title(image_info["file_name"], fontsize=8)
        if ann:
            segmentation = ann["segmentation"][0]
            points = list(zip(segmentation[0::2], segmentation[1::2]))
            ax.add_patch(Polygon(points, closed=True, fill=False, edgecolor="yellow", linewidth=1.5))
            ax.text(
                points[0][0],
                points[0][1],
                categories.get(ann["category_id"], str(ann["category_id"])),
                color="black",
                fontsize=8,
                bbox={"facecolor": "yellow", "edgecolor": "none", "pad": 1},
            )

    plt.tight_layout()
    plt.show()


visualize_prepared_samples(samples=6)

In [ ]:
ZIP_OUTPUT_DIR = WORKING_DIR / "artifacts"
ZIP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

zip_base_path = ZIP_OUTPUT_DIR / f"realwaste_normalized_{SEGMENTATION_MODEL}"
zip_path = shutil.make_archive(
    base_name=str(zip_base_path),
    format="zip",
    root_dir=OUTPUT_DIR.parent,
    base_dir=OUTPUT_DIR.name,
)

print("RealWaste result folder:", OUTPUT_DIR)
print("Zip file ready:", zip_path)
